# 0. Set Up

---
### 0.1 Tools: 
- `numpy` - time array and math
- `pandas` - cleaning and filtering the NASA catalog
- `matplotlib` & `matplotlib.patches` - all plots including the disk visualizations
- `ipywidgets` - interactive slider
- `astropy.units` - for constants and unit converstions to prevent unit errors

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display
from astropy import units as u

print("Libraries loaded successfully!")

Libraries loaded successfully!


---
### 0.2 Physical Constants: 

$D_{Moon} = 3,474.8 \text{ km}, \text{    }D_{Sun} = 1,392,700 \text{ km}$

$d_{Moon,0} = 384,400 \text{ km}, \text{    }d_{Sun} = 1.496*10^8 \text{ km}$

$v_{recession} = 3.8 \text{ cm/yr} = 3.8*10^{-5} \text{km/yr}$

$d_{Moon}(t) = d_0 + v * t, \text{    }R(t) = \frac{D_{Moon}/d_{Moon}(t)}{D_{Sun}/d_{Sun}}$

In [ ]:
# ── Physical constants ────────────────────────────────────────────────────────

D_moon   = 3_474.8       * u.km         # Moon diameter
D_sun    = 1_392_700.0   * u.km         # Sun diameter
d_moon_0 = 384_400.0     * u.km         # Moon semi-major axis (mean distance)
d_sun    = 149_597_870.0 * u.km         # Mean Earth-Sun distance (1 AU)

e_moon = 0.0549                         # Moon's orbital eccentricity (dimensionless)
b_moon_0 = d_moon_0 * np.sqrt(1 - e_moon**2)  # semi-minor axis at present

# Recession rate defined in natural observational units, converted to km/yr
v_recession = (3.8 * u.cm / u.yr).to(u.km / u.yr)

# Reference year ("t = 0" in our model)
YEAR_0 = 2026

print(f"Moon diameter:         {D_moon}")
print(f"Sun diameter:          {D_sun}")
print(f"Earth-Moon semi-major: {d_moon_0}")
print(f"Earth-Moon semi-minor: {b_moon_0:.1f}")
print(f"Earth-Sun distance:    {d_sun.to(u.au):.6f}")
print(f"Recession rate:        {v_recession:.2e}")

---
# 1 Calculations


### 1.1 Angular Size Model

The small-angle approximation gives angular diameter:
$$\theta \approx \frac{D}{d}$$

The Moon's distance grows linearly with time due to tidal recession:
$$d_\text{Moon}(t) = d_0 + v \cdot (t - t_0)$$

The angular size ratio we care about is:
$$R(t) = \frac{\theta_\text{Moon}(t)}{\theta_\text{Sun}} = \frac{D_\text{Moon} / d_\text{Moon}(t)}{D_\text{Sun} / d_\text{Sun}}$$

Total solar eclipses are only possible when $R(t) \geq 1$.

In [ ]:
# ── Time array: from 500 Myr ago to 1.5 Gyr in the future ────────────────────
years = np.linspace(-500_000_000, 1_500_000_000, 10_000) * u.yr

def moon_distance(t_offset):
    """Earth-Moon distance at t_offset (astropy Quantity in time units) from YEAR_0."""
    return d_moon_0 + v_recession * t_offset

def moon_semiminor(t_offset):
    """Semi-minor axis b = a*sqrt(1-e²) at t_offset from YEAR_0."""
    return moon_distance(t_offset) * np.sqrt(1 - e_moon**2)

def angular_size(diameter, distance):
    """Angular diameter (dimensionless, radians) via small-angle approx: theta = D/d."""
    return (diameter / distance).decompose()

def ratio_R(t_offset):
    """Moon/Sun angular size ratio at t_offset from YEAR_0."""
    theta_moon = angular_size(D_moon, moon_semiminor(t_offset))
    theta_sun  = angular_size(D_sun, d_sun)
    return theta_moon / theta_sun

# Compute arrays
d_moon_arr     = moon_distance(years)
theta_moon_arr = angular_size(D_moon, d_moon_arr)
R_arr          = ratio_R(years)

# ── Find the crossover year (R = 1) ──────────────────────────────────────────
# D_moon / (d0 + v*t) / sqrt(1-e²) = D_sun / d_sun  →  solve for t
t_crossover   = ((D_moon * d_sun / (D_sun * np.sqrt(1 - e_moon**2))) - d_moon_0) / v_recession
year_crossover = YEAR_0 + t_crossover.to(u.yr).value

print(f"Current R (today):        {ratio_R(0 * u.yr):.4f}")
print(f"Crossover in:             {t_crossover.to(u.Myr):.1f}")
print(f"Crossover year (approx):  {year_crossover:.2e}")

---
## 1.2 NASA Eclipse Catalog — Data Loading & Cleaning

Data source: [NASA Five Millennium Canon of Solar Eclipses](https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEcatalog.txt)

In [6]:
# ── Load NASA catalog ─────────────────────────────────────────────────────────
# Try local file first; fall back to URL

import os

LOCAL_PATH = "5MKSEcatalog.txt"
NASA_URL   = "https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEcatalog.txt"

source = LOCAL_PATH if os.path.exists(LOCAL_PATH) else NASA_URL
print(f"Loading from: {source}")

# Column widths for the fixed-width format (inspect header to confirm)
# Format reference: https://eclipse.gsfc.nasa.gov/5MCSE/5MCSEcat.html
col_names = [
    "Cat_Num", "Cat_Plate", "Year", "Month", "Day",
    "Greatest_Eclipse", "DT_s", "Luna_Num", "Saros_Num",
    "Ecl_Type", "QLE", "Gamma", "Ecl_Mag",
    "Lat", "Long", "Sun_Alt", "Sun_Azm",
    "Path_Width_km", "Central_Line_Dur"
]

df_raw = pd.read_fwf(
    source,
    skiprows=10,
    header=None,
    encoding="latin1"
)

# Assign column names (trim to actual number of columns)
df_raw.columns = col_names[:len(df_raw.columns)]

df_raw.head()

Loading from: https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEcatalog.txt


,Cat_Num,Cat_Plate,Year,Month,Day,Greatest_Eclipse,DT_s,Luna_Num,Saros_Num,Ecl_Type,QLE,Gamma,Ecl_Mag,Lat,Long,Sun_Alt,Sun_Azm,Path_Width_km,Central_Line_Dur
0,1,1,-1999,Jun,12,03:14:51,46438,-49456,5,T,-n,-0.2701,1.0733,6.0N,33.3W,74,344,247,06m37s
1,2,1,-1999,Dec,5,23:45:23,46426,-49450,10,A,n-,-0.2317,0.9382,32.9S,10.8E,76,21,236,06m44s
2,3,1,-1998,Jun,1,18:09:16,46415,-49444,15,T,p-,0.4994,1.0284,46.2N,83.4E,60,151,111,02m15s
3,4,1,-1998,Nov,25,05:57:03,46403,-49438,20,A,p-,-0.9045,0.9806,67.8S,143.8W,25,74,162,01m14s
4,5,1,-1997,Apr,22,13:19:56,46393,-49433,-13,P,-t,-1.4670,0.1611,60.6S,106.4W,0,281,NaN,NaN


# 1. Plot 1 & 2: Ratio and Angular Size Over Time

Standard `matplotlib` line plots. 

`plt.axhline(y=1, linestyle='-', color='red')` shows critical threshold
`plt.fill_between()` to shade the "annular only" future in a different color. 

This makes the science immediately obvious visually. 

# 2. Plot 3: Superimposed Sun/Moon Disks

Use `matplotlib.patches.Circle` - draw two cricles, scale their radii to the actual angular sizes at a given time `t`. 

Optional user-input feature: `ipywidgets` and specifically `widgets.FloatSlider` to add interactive year slider directly into the notebook
User drags the slider and watches the Moon shrink relative to the Sun in real time. 

# 3. Historical Eclipse Data Scatter from NASA

A scatter plot color coded by eclipse type (T = total, A = annular, H = hybrid) overlaid on the model curve.